In [ ]:
#| default_exp channel

In [ ]:
#| export
# channel.py

from __future__ import annotations

from abc import ABC, abstractmethod

from comm_core.core import Transmission, TransmissionResult
from comm_core.topology import Link


    

In [ ]:
#| export

class Channel(ABC):

    @abstractmethod
    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:
        pass

    def reset(self) -> None:
        pass

    def update(self, time: float) -> None:
        pass



In [ ]:
#| export

class IdentityChannel(Channel):
    """
    Perfect communication.
    """

    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:

        message = transmission.message

        return TransmissionResult(
            success=True,
            transmission=transmission,
            payload=message.payload,
            latency=0.0,
            size_bits=message.size_bits or 0,
            snr_db=float("inf"),
            error_rate=0.0,
        )




In [ ]:
#| export

class AWGNChannel(Channel):
    """
    Simple analytical channel.

    This intentionally models only packet success/failure
    rather than individual bits.
    """

    def __init__(
        self,
        snr_db: float,
        packet_error_rate: float = 0.0,
    ):
        self.snr_db = snr_db
        self.packet_error_rate = packet_error_rate

    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:

        import random

        success = (
            random.random()
            >= self.packet_error_rate
        )

        message = transmission.message

        return TransmissionResult(
            success=success,
            transmission=transmission,
            payload=(
                message.payload
                if success
                else None
            ),
            latency=0.0,
            size_bits=message.size_bits or 0,
            snr_db=self.snr_db,
            error_rate=self.packet_error_rate,
        )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()